## 역할 : Agent 실행 과정과 결과를 단계별로 확인

## STEP 1. Agent 초기화
Goal, Current State, Available Actions, History를 정의합니다.

In [1]:
from agent_core import ask_llm, execute_action

# 사용자 목표
goal = "다음 주 부산 출장 준비"

# 현재 상태
current_state = {
  "trip_date": None,
  "meeting_time": None,
  "transport_status": None,  
}

# 사용 가능한 도구
available_actions = [
  "check_calendar", 
  "search_transport", 
  "search_hotel"
]

# 실행 이력
history = []

## STEP 2. 첫 번째 LLM 판단
출장 날짜가 없는 상태에서 LLM이 다음 행동을 선택합니다.

In [2]:
decision = ask_llm(
  goal, 
  current_state, 
  available_actions, 
  history
)

print("Selected Action:", decision["action"])
print("Reason:", decision["reason"])

===== LLM에 전달하는 Context =====
{"goal": "다음 주 부산 출장 준비", "current_state": {"trip_date": null, "meeting_time": null, "transport_status": null}, "available_actions": ["check_calendar", "search_transport", "search_hotel"], "history": []}
Selected Action: check_calendar
Reason: 출장 날짜가 아직 확인되지 않았으므로 먼저 캘린더를 확인해야 합니다.


In [3]:
decision

{'action': 'check_calendar',
 'arguments': {'transport_type': None},
 'reason': '출장 날짜가 아직 확인되지 않았으므로 먼저 캘린더를 확인해야 합니다.'}

## STEP 3. 첫 번째 도구 실행
LLM이 선택한 check_calendar를 실행합니다. 실행 결과는 상태와 이력에 반영됩니다.

In [4]:
result = execute_action(
  decision, 
  current_state, 
  history
)

print("Calendar Result:", result)
print("Updated State:", current_state)
print("History:", history)

Calendar Result: {'trip_date': '2026-10-08', 'meeting_time': '14:00'}
Updated State: {'trip_date': '2026-10-08', 'meeting_time': '14:00', 'transport_status': None}
History: [{'action': 'check_calendar', 'arguments': {'transport_type': None}, 'result': {'trip_date': '2026-10-08', 'meeting_time': '14:00'}}]


In [5]:
history

[{'action': 'check_calendar',
  'arguments': {'transport_type': None},
  'result': {'trip_date': '2026-10-08', 'meeting_time': '14:00'}}]

## STEP 4. LLM 두 번째 판단
변경된 Current State와 History를 전달하여 다음 행동을 선택합니다.

In [6]:
decision = ask_llm(
  goal, 
  current_state, 
  available_actions, 
  history
)

print("Selected Action:", decision["action"])
print("Reason:", decision["reason"])

===== LLM에 전달하는 Context =====
{"goal": "다음 주 부산 출장 준비", "current_state": {"trip_date": "2026-10-08", "meeting_time": "14:00", "transport_status": null}, "available_actions": ["check_calendar", "search_transport", "search_hotel"], "history": [{"action": "check_calendar", "arguments": {"transport_type": null}, "result": {"trip_date": "2026-10-08", "meeting_time": "14:00"}}]}
Selected Action: search_transport
Reason: 출장 날짜가 확인되었고 아직 교통편 검색 이력이 없으므로 KTX를 먼저 검색해야 합니다.


STEP 5. 두 번째 도구 실행

search_transport를 실행합니다. 이번에는 KTX가 매진된 상황입니다.

In [7]:
result = execute_action(
    decision,
    current_state,
    history
)

print("Transport Result:", result)

Transport Result: {'transport_type': 'KTX', 'transport_status': 'sold_out', 'transport_message': 'KTX 좌석이 매진되었습니다.'}


STEP 6. 상태와 실행 이력 확인

KTX 매진 결과가 Current State와 History에 어떻게 기록되었는지 확인합니다.

In [8]:
import json

print("[Current State]")
print(json.dumps(
  current_state,
  ensure_ascii=False,
  indent=2
))

print("\n[History]")
print(json.dumps(
  history,
  ensure_ascii=False,
  indent=2
))


[Current State]
{
  "trip_date": "2026-10-08",
  "meeting_time": "14:00",
  "transport_status": "sold_out",
  "transport_type": "KTX",
  "transport_message": "KTX 좌석이 매진되었습니다."
}

[History]
[
  {
    "action": "check_calendar",
    "arguments": {
      "transport_type": null
    },
    "result": {
      "trip_date": "2026-10-08",
      "meeting_time": "14:00"
    }
  },
  {
    "action": "search_transport",
    "arguments": {
      "transport_type": "KTX"
    },
    "result": {
      "transport_type": "KTX",
      "transport_status": "sold_out",
      "transport_message": "KTX 좌석이 매진되었습니다."
    }
  }
]


STEP 7. 세 번째 LLM 판단

KTX 매진 사실을 알고 있는 LLM에게 다시 다음 행동을 선택하게 합니다.

In [9]:
decision = ask_llm(
    goal,
    current_state,
    available_actions,
    history
)

print("Selected Action:", decision["action"])
print("Reason:", decision["reason"])

===== LLM에 전달하는 Context =====
{"goal": "다음 주 부산 출장 준비", "current_state": {"trip_date": "2026-10-08", "meeting_time": "14:00", "transport_status": "sold_out", "transport_type": "KTX", "transport_message": "KTX 좌석이 매진되었습니다."}, "available_actions": ["check_calendar", "search_transport", "search_hotel"], "history": [{"action": "check_calendar", "arguments": {"transport_type": null}, "result": {"trip_date": "2026-10-08", "meeting_time": "14:00"}}, {"action": "search_transport", "arguments": {"transport_type": "KTX"}, "result": {"transport_type": "KTX", "transport_status": "sold_out", "transport_message": "KTX 좌석이 매진되었습니다."}}]}
Selected Action: search_transport
Reason: 출장 날짜가 확인되었고 KTX가 매진되었으므로, 동일 검색을 반복하지 않고 BUS를 검색해야 합니다.


In [10]:
print(json.dumps(
  decision,
  indent=2
))

{
  "action": "search_transport",
  "arguments": {
    "transport_type": "BUS"
  },
  "reason": "\ucd9c\uc7a5 \ub0a0\uc9dc\uac00 \ud655\uc778\ub418\uc5c8\uace0 KTX\uac00 \ub9e4\uc9c4\ub418\uc5c8\uc73c\ubbc0\ub85c, \ub3d9\uc77c \uac80\uc0c9\uc744 \ubc18\ubcf5\ud558\uc9c0 \uc54a\uace0 BUS\ub97c \uac80\uc0c9\ud574\uc57c \ud569\ub2c8\ub2e4."
}


STEP 8. 세 번째 도구 실행 - BUS 검색
- LLM이 선택한 교통수단을 확인하고 실행합니다.

In [11]:
print("Selected Action:", decision["action"])
print("Arguments:", decision["arguments"])

result = execute_action(
  decision,
  current_state,
  history
)

print("Transport Result:", result)

Selected Action: search_transport
Arguments: {'transport_type': 'BUS'}
Transport Result: {'transport_type': 'BUS', 'transport_status': 'available', 'transport_message': '버스 좌석이 있습니다.'}


STEP 9. 상태와 실행 이력 확인
- 현재 상태에서 BUS 검색 결과가 반영되고, 실행 이력에는 KTX 매진과 BUS 검색 성공이 모두 남는다.

In [12]:
print("[Current State]")
print(json.dumps(
    current_state,
    ensure_ascii=False,
    indent=2
))

print("\n[History]")
print(json.dumps(
    history,
    ensure_ascii=False,
    indent=2
))

[Current State]
{
  "trip_date": "2026-10-08",
  "meeting_time": "14:00",
  "transport_status": "available",
  "transport_type": "BUS",
  "transport_message": "버스 좌석이 있습니다."
}

[History]
[
  {
    "action": "check_calendar",
    "arguments": {
      "transport_type": null
    },
    "result": {
      "trip_date": "2026-10-08",
      "meeting_time": "14:00"
    }
  },
  {
    "action": "search_transport",
    "arguments": {
      "transport_type": "KTX"
    },
    "result": {
      "transport_type": "KTX",
      "transport_status": "sold_out",
      "transport_message": "KTX 좌석이 매진되었습니다."
    }
  },
  {
    "action": "search_transport",
    "arguments": {
      "transport_type": "BUS"
    },
    "result": {
      "transport_type": "BUS",
      "transport_status": "available",
      "transport_message": "버스 좌석이 있습니다."
    }
  }
]
